# The goals of this notebook:

The goal of this notebook is to add FIPS codes for each NOAA event, based on the path of the event (given by begin/end latitude and longitude) when available, or identification of the NWS Public Forecast Zone within each county.

A list of FIPS codes can be found at: https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt or https://datahub.transportation.gov/Railroads/State-County-and-City-FIPS-Reference-Table/eek5-pv8d/data_preview

To identify counties based on the (linear) path of an event, we can use a high-resolution shapefile of US counties from the US Census, available from https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html.

To identify NWS zones we can use a Zone-county Correlation File from https://www.weather.gov/gis/ZoneCounty (we could also try to use a shapefile from the National Weather Service, available from https://www.weather.gov/gis/PublicZones, but these zone numbers don't appear to match what are in the NOAA file).

We will then also add variables to the NOAA data to reflect the severity of the event. This includes:
- The overall duration of the event
- Predictors that are already in the NOAA data
- Maybe some ERA5 data, if we can figure out how to merge

Then, we'll create a new version of the data in which each event that occurred in multiple counties is split into a single line for the event in each county. We'll add a "duration per county" variable.

**Finally, we'll convert these data into a time series based on the start and endtimes of each event.**

First, load the NOAA data and then convert the state names to abbreviations

In [4]:
import pandas as pd
import geopandas

In [5]:
df_events = pd.read_csv("../Data/NOAA_StormEvents/StormEvents_2014_2024.csv")

In [6]:
# We'll use a dictionary to convert state names to abbreviations
# (There is a Pandas 'us' package that could do this for us, but I'm having dependency issues, so a dictionary it is...)

us_state_to_abbrev = {
    "ALABAMA": "AL",
    "ALASKA": "AK",
    "ARIZONA": "AZ",
    "ARKANSAS": "AR",
    "CALIFORNIA": "CA",
    "COLORADO": "CO",
    "CONNECTICUT": "CT",
    "DELAWARE": "DE",
    "FLORIDA": "FL",
    "GEORGIA": "GA",
    "HAWAII": "HI",
    "IDAHO": "ID",
    "ILLINOIS": "IL",
    "INDIANA": "IN",
    "IOWA": "IA",
    "KANSAS": "KS",
    "KENTUCKY": "KY",
    "LOUISIANA": "LA",
    "MAINE": "ME",
    "MARYLAND": "MD",
    "MASSACHUSETTS": "MA",
    "MICHIGAN": "MI",
    "MINNESOTA": "MN",
    "MISSISSIPPI": "MS",
    "MISSOURI": "MO",
    "MONTANA": "MT",
    "NEBRASKA": "NE",
    "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH",
    "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM",
    "NEW YORK": "NY",
    "NORTH CAROLINA": "NC",
    "NORTH DAKOTA": "ND",
    "OHIO": "OH",
    "OKLAHOMA": "OK",
    "OREGON": "OR",
    "PENNSYLVANIA": "PA",
    "RHODE ISLAND": "RI",
    "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN",
    "TEXAS": "TX",
    "UTAH": "UT",
    "VERMONT": "VT",
    "VIRGINIA": "VA",
    "WASHINGTON": "WA",
    "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI",
    "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC",
    "AMERICAN SAMOA": "AS",
    "GUAM": "GU",
    "NORTHERN MARIANA ISLANDS": "MP",
    "PUERTO RICO": "PR",
    "UNITED STATES MINOR OUTLYING ISLANDS": "UM",
    "U.S": "US"
}

def convert_state_name_to_abbrev(state_name):
    #Note that the state name needs to be in all capital letters
    return us_state_to_abbrev.get(state_name, "Unknown")

# Apply the function to convert the STATE variable in df_events to abbreviations
df_events['STATE_ABBREV'] = df_events['STATE'].apply(convert_state_name_to_abbrev)

Many (about 16,000) of these observations are from bodies of water/oceans or from US territories or protectorates. We'll drop these data.

In [7]:
#Drop rows where STATE_ABBREV is either Unknown, AS, GU, MP, PR, UM, or US
df_events = df_events[~df_events['STATE_ABBREV'].isin(["Unknown", "AS", "GU", "MP", "PR", "UM", "US"])]

Next, load the US Counties shapefile from the US Census.

In [8]:
#Load the US Census Counties shapefile
counties = geopandas.read_file('../Data/County_Shapefiles/cb_2023_us_county_500k')

#Concatenate STATEFP and COUNTFP and then convert to an integer
counties['FIPS'] = (counties['STATEFP'].astype(str) + counties['COUNTYFP'].astype(str)).astype(int)

Define a function that identifies all the FIPS in the path of the weather event.

We're going to assume that the path of the event is essentially linear. This is probably inaccurate, but we don't have any additional data (without doing something really clever with the ERA5 data) as an alternative

In [ ]:
# Given a beginning point (given by BEGIN_LAT and BEGIN_LON) and an ending point (given by END_LAT and END_LON) from df_events, 
# identify which FIPS values from counties lie in the path between the beginning and ending points

def get_fips_from_path(row):
    #Get the beginning and ending values of longitude and latitude
    begin = (row['BEGIN_LON'], row['BEGIN_LAT'])
    end = (row['END_LON'], row['END_LAT'])

    #Convert them into geopandas points
    points = geopandas.points_from_xy([begin[0], end[0]], [begin[1], end[1]])

    #Get the path between the two points
    path = geopandas.GeoSeries(points)
    
    #Get the FIPS values from the counties shapefile that intersect with the path
    fips = counties[counties.geometry.intersects(path.union_all())]['FIPS'].tolist()
    
    return fips

#Apply get_fips_from_path to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Path'] = df_events.apply(get_fips_from_path, axis=1)

The NOAA data includes columns for STATE_FIPS and CZ_FIPS. When the CZ_TYPE variable is C (i.e., County/Parish), these values appear to match "official lists."

However, when CZ_TYPE is Z, these appear to refer to a NWS forecast zone. These zones often correspond to counties, but can also correspond to sub-areas of counties 
that might experience different weather patterns (e.g., a particularly tall mountain).

The NWS has a zone-county correlation file that we can use to look up the FIPS code based on a concatenation of the state abbreviation and the CZ_FIPS value for *most* of the values. For others, we'll need to resort to some other scraping.

In [10]:
#Import the zone-county correlation file
df_zonecountycorr = pd.read_csv('../Data/NOAA_Cleaned_Data/bp05mr24.dbx', sep="|")

#Add the column headings
df_zonecountycorr.columns = ['STATE', 'ZONE', 'CWA', 'NAME', 'STATE_ZONE', 'COUNTY', 'FIPS', 'TIME_ZONE', 'FE_AREA', 'LAT', 'LON']

#Create a new variable in df_events by first adding zeroes to make CZ_FIPS a three-digit number and then concatenating with STATE_ABBREV
df_events['CZ_FIPS_ZONE'] = df_events['STATE_ABBREV'] + df_events['CZ_FIPS'].apply(lambda x: str(x).zfill(3))

#Merge the FIPS column from df_zonecountycorr with df_events, matching on CZ_FIPS_ZONE from df_events and STATE_ZONE from df_zonecountycorr
df_events = df_events.merge(df_zonecountycorr[['STATE_ZONE', 'FIPS']], left_on='CZ_FIPS_ZONE', right_on='STATE_ZONE', how='left')

#Rename the FIPS column FIPS_from_Zone
df_events.rename(columns={'FIPS': 'FIPS_from_Zone'}, inplace=True)

#Drop the CZ_FIPS_ZONE and STATE_ZONE variables from df_events
df_events.drop(columns=['CZ_FIPS_ZONE', 'STATE_ZONE'], inplace=True)

There are still some FIPS codes that aren't being identified. We'll try to match the CZ_NAME value in df_events with the NAME variable in df_zonecountycorr.

The two sets of names appear to be similar, but not always exact. So we'll need to do some fuzzy matching. We'll try the Jaro-Winkler algorithm (from jellyfish)

Note that running this code on the entire dataframe would take nearly 35 minutes. But we can just run it on rows that don't yet have a FIPS code.

In [11]:
import jellyfish

# For each row in df_events, use its STATE_ABBREV to create a new data frame from matching values of the STATE variable in df_zonecountycorr
# Then use Jaro-Winkler to do a fuzzy match on the CZ_NAME variable in df_events with the NAME variable in the new data frame. 
# For the best match, import the FIPS value from df_zonecountycorr

def get_fips_from_name(row):
    #Get the state abbreviation
    state_abbrev = row['STATE_ABBREV']
    
    #Create a new data frame from matching values of the STATE variable in df_zonecountycorr
    df_state = df_zonecountycorr[df_zonecountycorr['STATE'] == state_abbrev]
    
    #Get the CZ_NAME from the row
    cz_name = row['CZ_NAME']
    
    #Get the best match using jaro_winkler
    best_match = None
    best_score = 0
    best_fips = None
    
    for index, row in df_state.iterrows():
        score = jellyfish.jaro_winkler_similarity(cz_name, row['NAME'])
        if score > best_score:
            best_score = score
            best_match = row['NAME']
            best_fips = row['FIPS']
    
    return best_fips

#Apply get_fips_from_name to each row of df_events where the length of FIPS_from_Path is 0 and where FIPS_from_Zone is NaN and create a new variable that lists the fips values
df_events['FIPS_from_Name'] = df_events.apply(lambda row: get_fips_from_name(row) if len(row['FIPS_from_Path']) == 0 and pd.isna(row['FIPS_from_Zone']) else None, axis=1)

Now, we'll create a list of all the FIPS associated with each event. For instances where we know the path, we'll default to that. If we don't know the path, then we'll use FIPS_from_Zone; if we don't have either, then we'll use FIPS_from_Name

In [12]:
#to ensure that our FIPS values are all lists, we'll first transform FIPS_from_Zone and FIPS_from_Name into lists
df_events['FIPS_from_Zone'] = df_events['FIPS_from_Zone'].apply(lambda x: [x] if pd.notna(x) else [])
df_events['FIPS_from_Name'] = df_events['FIPS_from_Name'].apply(lambda x: [x] if pd.notna(x) else [])

#Then we'll create a new variable that is equal to FIPS_from_Path if that's not empty; if it is we'll use FIPS_from_Zone and then FIPS_from_Name
df_events['FIPS'] = df_events.apply(lambda x: x['FIPS_from_Path'] if len(x['FIPS_from_Path']) > 0 else (x['FIPS_from_Zone'] if pd.notna(x['FIPS_from_Zone']) else x['FIPS_from_Name']), axis=1)

/var/folders/gs/h69vjsl512b8200d29lxpgrc0000gn/T/ipykernel_49560/3395774983.py:6: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  df_events['FIPS'] = df_events.apply(lambda x: x['FIPS_from_Path'] if len(x['FIPS_from_Path']) > 0 else (x['FIPS_from_Zone'] if pd.notna(x['FIPS_from_Zone']) else x['FIPS_from_Name']), axis=1)


In [13]:
#Add a new variable that is equal to the length of FIPS
df_events['Number_of_FIPS'] = df_events['FIPS'].apply(len)

## Fixing Timestamps

Timestamps are in local time; we need to conver them to UTC

It looks like each timezone name includes the difference between it and UTC. i.e.:
'EST-5', 'CST-6', 'PST-8', 'MST-7', 'HST-10', 'AKST-9', 'PDT-7', 'CDT-5', 'EDT-4

In [14]:
# Convert BEGIN_DATE_TIME and END_DATE_TIME to UTC using CZ_DATETIME values of 'EST-5', 'CST-6', 'PST-8', 'MST-7', 'HST-10', 'AKST-9', 'PDT-7', 'CDT-5', 'EDT-4'
def convert_to_utc(row):
    #Get the CZ_DATETIME value
    cz_timezone = row['CZ_TIMEZONE']
    begin_date_time = row['BEGIN_DATE_TIME']
    end_date_time = row['END_DATE_TIME']
    
    #Get the timezone offset
    timezone_offset = cz_timezone.split('-')[1]
    timezone_offset = int(timezone_offset)

    # Return BEGIN_DATE_TIME and END_DATE_TIME minus pd.Timedelta(hours=timezone_offset)
    return pd.to_datetime(begin_date_time) + pd.Timedelta(hours=timezone_offset), pd.to_datetime(end_date_time) + pd.Timedelta(hours=timezone_offset)

#Apply the function to convert the BEGIN_DATE_TIME and END_DATE_TIME variables in df_events to UTC
df_events['BEGIN_DATE_TIME_UTC'], df_events['END_DATE_TIME_UTC'] = zip(*df_events.apply(convert_to_utc, axis=1))


There are a bunch of variables we don't need. We'll drop them here.

In [15]:
df_events.drop(columns=['BEGIN_YEARMONTH', 'BEGIN_DAY', 'BEGIN_TIME',
                        'END_YEARMONTH', 'END_DAY', 'END_TIME',
                        'EPISODE_ID', 'EVENT_ID',
                        'STATE', 'STATE_FIPS',
                        'YEAR', 'MONTH_NAME',
                        'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'SOURCE',
                        'FLOOD_CAUSE', 'CATEGORY',
                        'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO', 'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME',
                        'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE', 'END_AZIMUTH', 'END_LOCATION', 'DATA_SOURCE'], inplace=True)

#We also no longer heed the FIPS_from_Path, FIPS_from_Zone, and FIPS_from_Name variables
df_events.drop(columns=['FIPS_from_Path', 'FIPS_from_Zone', 'FIPS_from_Name'], inplace=True)

There are several ways we could measure the severity of each event. One basic way is to compute the duration of the event based on the begin year/day/time (measured by the BEGIN_DATE_TIME variable) and end year/day/time (measured by the END_DATE_TIME variable). 

In [16]:
#Create a new variable in df_events total_duration by subtracting the BEGIN_DATE_TIME from END_DATE_TIME, measuring in minutes.
#Specify the format day-month-year hour:minute:second
df_events['BEGIN_DATE_TIME_UTC'] = pd.to_datetime(df_events['BEGIN_DATE_TIME_UTC'], format='%d-%b-%y %H:%M:%S')
df_events['END_DATE_TIME_UTC'] = pd.to_datetime(df_events['END_DATE_TIME_UTC'], format='%d-%b-%y %H:%M:%S')
df_events['total_duration_min'] = (df_events['END_DATE_TIME_UTC'] - df_events['BEGIN_DATE_TIME_UTC']).dt.total_seconds() / 60.0

For events that involve multiple counties, we could assume that either (1) the event is localized enough that it only impacts one county at a time, or (2) it affects multiple counties simultaneously. If we assume #1, then we might want to divide the total duration by the number of counties in the path of the event.

In [17]:
#Divide total_duration_min by the number of items in the FIPS list
df_events['total_duration_perfips_min'] = df_events['total_duration_min'] / df_events['Number_of_FIPS']

Now we'll start exporting these data. At first, we'll keep things basically as they are with one row per event, regardless of the number of counties in which the event takes place.

To enable merging, we'll associate each event with the first FIPS value in its list.

Then we'll export as a csv

In [18]:
#Create a new variable FIPS_First that is the first value in the list of each FIPS value
df_events['FIPS_First'] = df_events['FIPS'].apply(lambda x: x[0] if len(x) > 0 else None).astype(int)

The next thing we'd like to do is to associate each event with a power grid subregion. This should generally be straightforward--most of these events occurred in a single county/FIPS value. But some of the larger events could span multiple counties and, potentially, multiple subregions.

In [19]:
#Load the Counties_2020.csv file
df_counties_2020 = pd.read_csv('../Data/County_level_Variables/Counties_2020.csv')

#Merge df_events with the Subregion variable from df_counties using FIPS_First from df_events and FIPS from df_counties_2020
df_events = df_events.merge(df_counties_2020[['FIPS', 'Subregion']], left_on='FIPS_First', right_on='FIPS', how='left')

#Remove the variable FIPS_y
df_events.drop(columns=['FIPS_y'], inplace=True)

#Rename FIPS_x as FIPS
df_events.rename(columns={'FIPS_x': 'FIPS'}, inplace=True)

Next, we'll split each multi-FIPS event into separate rows. Then we can look up the eGRID subregion for the FIPS, and then export

In [20]:
#For each row in df_events where Number_of_FIPS >1, make one copy of that row for each item in the FIPS list
df_events_exploded = df_events.explode('FIPS')

#Drop the FIPS_First and Subregion variables
df_events_exploded.drop(columns=['FIPS_First', 'Subregion'], inplace=True)

#Merge df_events with the Subregion variable from df_counties using FIPS_First from df_events and FIPS from df_counties_2020
df_events_exploded = df_events_exploded.merge(df_counties_2020[['FIPS', 'Subregion']], left_on='FIPS', right_on='FIPS', how='left')

# Time Series Conversion

To look for connections between the NOAA data and the eaglei data, we'll need to convert the NOAA data into a time series

We can either create a separate time series for each FIPS, or we could create a time series with a multiindex of time and FIPS.

There are over 50 types of weather events. We could look at these individually or combine them into categories.

In [30]:
#Start by defining the event categories
snowice_types = ['Heavy Snow', 'Winter Storm', 'Winter Weather', 'Ice Storm', 'Extreme Cold/Wind Chill','Blizzard', 'Avalanche', 'Cold/Wind Chill', 'Frost/Freeze','Sleet','Freezing Fog', 'Lake-Effect Snow']
flood_types = ['Flash Flood', 'Coastal Flood','Lakeshore Flood','Debris Flow']
storm_types = ['Heavy Rain','Tropical Storm','Tropical Depression','Hail','Lightning','Marine Lightning']
hurricane_types = ['Hurricane','Hurricane (Typhoon)']
heat_types = ['Excessive Heat', 'Heat']
fire_types= ['Wildfire','Dense Smoke']
wind_types = ['High Wind', 'Strong Wind', 'Thunderstorm Wind', 'Marine Thunderstorm Wind','Tornado','Waterspout', 'Funnel Cloud']
ocean_types = ['Marine Hurricane/Typhoon','Rip Current','Astronomical Low Tide', 'Marine Dense Fog','Marine Tropical Depression','Marine Strong Wind', 'Marine High Wind', 'Marine Hail','High Surf', 'Marine Tropical Storm', 'Seiche','Storm Surge/Tide', 'Tsunami', 'Sneakerwave']
other_types = ['Drought','Dust Storm', 'Dense Fog', 'Dust Devil', 'Volcanic Ashfall']

#Create a new variable in df_events that identifies the list that EVENT_TYPE is in
df_events['EVENT_CATEGORY'] = df_events['EVENT_TYPE'].apply(lambda x: 'Snow/Ice' if x in snowice_types else ('Flood' if x in flood_types else ('Storm' if x in storm_types else ('Hurricane' if x in hurricane_types else ('Heat' if x in heat_types else ('Fire' if x in fire_types else ('Wind' if x in wind_types else ('Ocean' if x in ocean_types else ('Other' if x in other_types else 'Unknown')))))))))
df_events_exploded['EVENT_CATEGORY'] = df_events_exploded['EVENT_TYPE'].apply(lambda x: 'Snow/Ice' if x in snowice_types else ('Flood' if x in flood_types else ('Storm' if x in storm_types else ('Hurricane' if x in hurricane_types else ('Heat' if x in heat_types else ('Fire' if x in fire_types else ('Wind' if x in wind_types else ('Ocean' if x in ocean_types else ('Other' if x in other_types else 'Unknown')))))))))

The competition utilities provided a function to construct a time series for a given state.

We'll start by adapting it to produce a time series for a given FIPS

In [ ]:
# The function below is adapted from the provided utilities and makes a time series for a given fips

def make_ts_events(fips, event_categories, start_year, start_month, start_day, end_year, end_month, end_day, df):
    """
    Construct a DataFrame with 6-hour intervals indicating event occurrence.
   
    Parameters:
    - df (pd.DataFrame): The NOAA StormEvent database.
    - fips (int/float): The FIPS to filter.
    - event_categories (list): The event categories to filter (e.g., ["Storm", "Hurricane"]).
    - start_year (int): The start year for the new DataFrame.
    - start_month (int): The start month for the new DataFrame.
    - start_day (int): The start day for the new DataFrame.
    - end_year (int): The end year for the new DataFrame.
    - end_month (int): The end month for the new DataFrame.
    - end_day (int): The end day for the new DataFrame.
   
    Returns:
    - pd.DataFrame: A DataFrame with 6-hour intervals and event counts.
    """

    # Convert the start and end times to a Pandas datetime object
    # Then create a new DataFrame with a time range from start_date to end_date with 6-hour intervals
    # This should include the last time interval - i.e., the one that starts at 18:00
    start_date = f"{start_year}-{start_month:02d}-{start_day:02d}"
    end_date = f"{end_year}-{end_month:02d}-{end_day:02d} 18:00"
    time_index = pd.date_range(start=start_date,
                                    end=end_date,
                                    freq='6h')
    

    
    # Create a new dataframe using time_index above as one of the variables
    new_df = pd.DataFrame({'time': time_index})

    # Initialize event count columns for each event type
    for category in event_categories:
        new_df[f'event_count {category}'] = 0  # Initialize event counts to 0
   
    # Filter the NOAA data for the specified FIPS and event type
    filtered_df = df[
        (df['FIPS'] == fips) & 
        (df['EVENT_CATEGORY'].isin(event_categories)) & 
        (df['END_DATE_TIME_UTC'] >= start_date) & 
        (df['BEGIN_DATE_TIME_UTC'] <= end_date)
    ].copy(deep=True)
   
    # Iterate through the events and assign them to the closest time interval in the new DataFrame
    for event_category in event_categories:
        event_subset = filtered_df[filtered_df['EVENT_CATEGORY']==event_category]
        
        for _, row in event_subset.iterrows():
            event_start = row['BEGIN_DATE_TIME_UTC']
            event_end = row['END_DATE_TIME_UTC']
       
            # Round the start and end times to the nearest 6-hour interval
            event_start_rounded = event_start.round('6h')
            event_end_rounded = event_end.round('6h')
       
            # Find the indices in the new DataFrame for the rounded times
            start_idx = new_df['time'].searchsorted(event_start_rounded)
            end_idx = new_df['time'].searchsorted(event_end_rounded)
       
            # Increment the event count for the affected time intervals
            if start_idx < len(new_df) and end_idx <= len(new_df):
                new_df.loc[start_idx:end_idx, f'event_count {event_category}'] += 1
    
    # Return the new_df
    return new_df   

Next, we can try to make a dataframe for all FIPS.

It will have a multiindex from the time and FIPS variables

If we want to merge with another array that has latitude and longitude as coordinates, we'll need to transform the FIPS into (centroid) latitude/longitude (we have these info in the counties dataframe)

In [32]:
event_categories = ['SnowIce', 'Flood', 'Storm', 'Hurricane', 'Heat', 'Fire', 'Wind', 'Ocean', 'Other']

# Create a list of all FIPS values in df_events_exploded
fips_list = df_events_exploded['FIPS'].unique()

# Iterate through fips_list, apply make_ts_events to each one, add a new variable equal to the value of fips
# and then concatenate the results into a single dataframe
all_dfs = []
for fips in fips_list:
    ts_df = make_ts_events(fips, event_categories, 2014, 1, 1, 2023, 12, 31, df_events_exploded)
    ts_df['FIPS'] = fips
    all_dfs.append(ts_df)

# Combine these into a single dataframe
df_timeseries = pd.concat(all_dfs, ignore_index=True)

# Set the index from the time and FIPS variables
df_timeseries.set_index(['time', 'FIPS'], inplace=True)

In [33]:
# Export df_timeseries to a parquet file
df_timeseries.to_parquet('../Data/NOAA_Cleaned_Data/NOAA_Timeseries.parquet', index=True)